# SMS spam optimised

The pipeline you built earlier for the SMS spam model used the default parameters for all of the elements in the pipeline. It's very unlikely that these parameters will give a particularly good model though. In this exercise you're going to run the pipeline for a selection of parameter values. We're going to do this in a systematic way: the values for each of the hyperparameters will be laid out on a grid and then pipeline will systematically run across each point in the grid.

In this exercise you'll set up a parameter grid which can be used with cross validation to choose a good set of parameters for the SMS spam classifier.

The following are already defined:

- `hasher` — a `HashingTF` object and
- logistic — a `LogisticRegression` object.

## Instructions

- Create a parameter grid builder object.
- Add grid points for `numFeatures` and `binary` parameters to the `HashingTF` object, giving values 1024, 4096 and 16384, and True and False, respectively.
- Add grid points for `regParam` and `elasticNetParam` parameters to the `LogisticRegression` object, giving values of 0.01, 0.1, 1.0 and 10.0, and 0.0, 0.5, and 1.0 respectively.
- Build the parameter grid.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [4]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
schema = StructType([
    StructField("id", IntegerType()),
	StructField("text", StringType()),
	StructField("label", IntegerType())
	])
	
sms = spark.read.csv("file:///home/talentum/test-jupyter/c5-MLWithPySpark/M4-EnsemblesAndPipelines/3_GridSearch/dataset/sms.csv", 
                     sep=';', header=False, schema=schema)
					 
from pyspark.sql.functions import regexp_replace
sms = sms.withColumn('text', regexp_replace(sms.text, '[_():;,.!?\\\\\\\\-]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '[0-9]', ' '))
sms = sms.withColumn('text', regexp_replace(sms.text, '(^ +| +$)', ''))
sms = sms.withColumn('text', regexp_replace(sms.text, ' +', ' '))

from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
tokenizer = Tokenizer(inputCol='text', outputCol='words')
remover = StopWordsRemover(inputCol=tokenizer.getOutputCol(), outputCol='terms')
hasher = HashingTF(inputCol=remover.getOutputCol(), outputCol="hash")
idf = IDF(inputCol=hasher.getOutputCol(), outputCol="features")
logistic = LogisticRegression()
pipeline = Pipeline(stages = [tokenizer, remover, hasher, idf, logistic])

from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
# Create parameter grid
params = ____()

# Add grid for hashing trick parameters
params = params.____(____, ____) \
               .____(____, ____)

# Add grid for logistic regression parameters
params = params.____(____, ____) \
               .____(____, ____)

# Build parameter grid
params = ____.____()